# 🤖 Notebook 3: Modelagem de Machine Learning

## 🎯 Objetivo deste Notebook

Este notebook implementa a fase de **Modeling** da metodologia CRISP-DM, desenvolvendo modelos preditivos para gestão de risco de IA em serviços financeiros.

### O que este notebook engloba:

1. **Preparação de Dados para ML**: Feature engineering, encoding e normalização
2. **Modelo 1 - Classificação de Severidade**: Prever nível de severidade de incidentes
3. **Modelo 2 - Predição de Investigação Regulatória**: Prever probabilidade de investigação
4. **Avaliação de Modelos**: Métricas de performance e validação cruzada
5. **Interpretabilidade**: Feature importance e SHAP values
6. **Salvamento de Modelos**: Persistência para uso na API

### Modelos implementados:
- Random Forest Classifier
- XGBoost Classifier  
- Logistic Regression (baseline)

---

## 📦 1. Importação de Bibliotecas

**O que faz**: Importa bibliotecas de machine learning, incluindo scikit-learn, XGBoost e ferramentas de interpretabilidade.

**Bibliotecas de ML**:
- `sklearn`: Modelos, métricas, preprocessing
- `xgboost`: Gradient boosting de alta performance
- `joblib`: Salvamento de modelos

In [ ]:
# Bibliotecas básicas
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Machine Learning - Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

# XGBoost
try:
    import xgboost as xgb
    print(f"✅ XGBoost {xgb.__version__} disponível")
except ImportError:
    print("⚠️ XGBoost não instalado. Instale com: !pip install xgboost")

# Salvamento de modelos
import joblib
import pickle

# Configurações
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
RANDOM_STATE = 42

print("\n✅ Bibliotecas importadas com sucesso!")
print(f"📌 Scikit-learn versão: {import sklearn; sklearn.__version__}")

## 📂 2. Carregamento e Preparação dos Dados

**O que faz**: Carrega o dataset processado e prepara features e targets para modelagem.

**Caso de uso**: Transformar dados brutos em formato adequado para algoritmos de ML.

In [ ]:
# Carregando dados
df = pd.read_csv('incidents_finance_filtered.csv')

print("="*80)
print("📊 DADOS CARREGADOS PARA MODELAGEM")
print("="*80)
print(f"\n✅ Total de incidentes: {len(df):,}")
print(f"📋 Total de features: {len(df.columns)}")

# Verificando valores ausentes
missing_summary = df.isnull().sum()[df.isnull().sum() > 0]
if len(missing_summary) > 0:
    print("\n⚠️ Valores ausentes encontrados:")
    print(missing_summary)
else:
    print("\n✅ Nenhum valor ausente nas colunas principais")

print("\n📊 Distribuição das variáveis target:")
print(f"\n1. severity_level:")
print(df['severity_level'].value_counts())
print(f"\n2. regulatory_investigation:")
print(df['regulatory_investigation'].value_counts())

## 🛠️ 3. Feature Engineering para ML

**O que faz**: Prepara features categóricas e numéricas para os algoritmos de ML.

**Técnicas aplicadas**:
- Label Encoding para variáveis categóricas ordinais
- One-Hot Encoding para variáveis categóricas nominais
- Criação de features numéricas derivadas

**Definição**: Feature engineering é o processo de criar ou transformar variáveis para melhorar a performance dos modelos.

**Caso de uso**: Algoritmos de ML geralmente funcionam melhor com features numéricas bem preparadas.

In [ ]:
# Selecionando features para modelagem
feature_columns = [
    'application_type',
    'incident_type', 
    'customer_segment',
    'year',
    'regulatory_investigation',
    'fine_imposed',
    'policy_change',
    'third_party_audit'
]

# Criando dataset de trabalho
df_ml = df[feature_columns + ['severity_level']].copy()

# Removendo linhas com valores ausentes (se houver)
df_ml = df_ml.dropna()

print("="*80)
print("🛠️ PREPARAÇÃO DE FEATURES")
print("="*80)
print(f"\n✅ Dataset para ML: {len(df_ml)} registros")
print(f"\n📋 Features selecionadas:")
for i, col in enumerate(feature_columns, 1):
    print(f"  {i}. {col}")

In [ ]:
# Encoding de variáveis categóricas com One-Hot Encoding
categorical_features = ['application_type', 'incident_type', 'customer_segment']

df_encoded = pd.get_dummies(df_ml, columns=categorical_features, drop_first=True)

print("\n" + "="*80)
print("🔢 ENCODING REALIZADO")
print("="*80)
print(f"\n✅ Features após encoding: {len(df_encoded.columns) - 1}")
print("\n📋 Novas features criadas:")
new_features = [col for col in df_encoded.columns if col not in feature_columns and col != 'severity_level']
for feat in new_features[:10]:  # Mostrando primeiras 10
    print(f"  • {feat}")
if len(new_features) > 10:
    print(f"  ... e mais {len(new_features) - 10} features")

In [ ]:
# Encoding do target (severity_level) para classificação
# Convertendo para binário: low/medium vs high/critical
df_encoded['severity_binary'] = df_encoded['severity_level'].apply(
    lambda x: 1 if x in ['high', 'critical'] else 0
)

# Também criar versão multiclasse com Label Encoding
le_severity = LabelEncoder()
severity_order = ['low', 'medium', 'high', 'critical']
df_encoded['severity_multiclass'] = df_encoded['severity_level'].apply(
    lambda x: severity_order.index(x) if x in severity_order else -1
)

print("\n" + "="*80)
print("🎯 TARGET ENCODING")
print("="*80)
print("\nTarget binário (severity_binary):")
print(df_encoded['severity_binary'].value_counts())
print("  0 = Low/Medium")
print("  1 = High/Critical")

print("\nTarget multiclasse (severity_multiclass):")
print(df_encoded['severity_multiclass'].value_counts().sort_index())
print("  0 = Low, 1 = Medium, 2 = High, 3 = Critical")

## 🎯 4. MODELO 1: Classificação de Severidade (Binária)

**Objetivo**: Prever se um incidente será de alta severidade (high/critical) ou baixa severidade (low/medium).

**Variável target**: `severity_binary` (0 = baixa, 1 = alta)

**Caso de uso real**: Gestores de risco podem usar este modelo para:
- Priorizar investigações de incidentes novos
- Alocar recursos de mitigação
- Estimar necessidade de capital regulatório (Basel)

### 4.1 Preparação dos Dados

In [ ]:
# Separando features (X) e target (y)
X = df_encoded.drop(['severity_level', 'severity_binary', 'severity_multiclass'], axis=1)
y = df_encoded['severity_binary']

print("="*80)
print("📊 PREPARAÇÃO PARA MODELO 1: Classificação de Severidade")
print("="*80)
print(f"\n📋 Dimensão de X (features): {X.shape}")
print(f"🎯 Dimensão de y (target): {y.shape}")
print(f"\n📊 Distribuição do target:")
print(y.value_counts())
print(f"\n⚖️ Proporção de classes:")
print(f"  Baixa severidade: {(y==0).sum()/len(y)*100:.1f}%")
print(f"  Alta severidade: {(y==1).sum()/len(y)*100:.1f}%")

# Verificando desbalanceamento
imbalance_ratio = (y==0).sum() / (y==1).sum()
if imbalance_ratio > 2 or imbalance_ratio < 0.5:
    print(f"\n⚠️ Dataset desbalanceado! Ratio: {imbalance_ratio:.2f}")
    print("   Considerar técnicas de balanceamento (SMOTE, class_weight)")
else:
    print(f"\n✅ Dataset relativamente balanceado (ratio: {imbalance_ratio:.2f})")

In [ ]:
# Split treino/teste (80/20) com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=y  # Mantém proporção de classes
)

print("\n" + "="*80)
print("✂️ SPLIT TREINO/TESTE")
print("="*80)
print(f"\n📚 Treino: {len(X_train)} amostras ({len(X_train)/len(X)*100:.1f}%)")
print(f"🧪 Teste: {len(X_test)} amostras ({len(X_test)/len(X)*100:.1f}%)")
print(f"\n📊 Distribuição no conjunto de treino:")
print(y_train.value_counts())
print(f"\n📊 Distribuição no conjunto de teste:")
print(y_test.value_counts())

### 4.2 Modelo Baseline: Logistic Regression

**O que faz**: Treina um modelo de regressão logística como baseline.

**Definição**: Regressão logística é um modelo linear para classificação que prediz probabilidades de classes.

**Por que baseline**: É importante ter um modelo simples como referência antes de testar modelos mais complexos.

**Caso de uso**: Além de performance, a regressão logística oferece interpretabilidade via coeficientes.

In [ ]:
print("="*80)
print("🔵 MODELO BASELINE: Logistic Regression")
print("="*80)

# Treinando modelo
lr_model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    class_weight='balanced'  # Lida com desbalanceamento
)

lr_model.fit(X_train, y_train)

# Predições
y_pred_lr = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Métricas
print("\n📊 MÉTRICAS DE PERFORMANCE:")
print(f"  • Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"  • Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"  • Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"  • F1-Score:  {f1_score(y_test, y_pred_lr):.4f}")
print(f"  • ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_lr):.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Baixa Severidade', 'Alta Severidade']))

### 4.3 Random Forest Classifier

**O que faz**: Treina um ensemble de árvores de decisão para classificação.

**Definição**: Random Forest combina múltiplas árvores de decisão treinadas em subsets aleatórios dos dados, reduzindo overfitting.

**Vantagens**:
- Robusto a outliers
- Lida bem com features categóricas
- Fornece feature importance
- Geralmente alta performance

**Caso de uso**: Modelo padrão para dados tabulares em produção.

In [ ]:
print("="*80)
print("🌲 RANDOM FOREST CLASSIFIER")
print("="*80)

# Treinando modelo com hiperparâmetros otimizados
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1  # Usa todos os cores
)

print("\n🔄 Treinando Random Forest...")
rf_model.fit(X_train, y_train)
print("✅ Treinamento concluído!")

# Predições
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Métricas
print("\n📊 MÉTRICAS DE PERFORMANCE:")
print(f"  • Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"  • Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"  • Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"  • F1-Score:  {f1_score(y_test, y_pred_rf):.4f}")
print(f"  • ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Baixa Severidade', 'Alta Severidade']))

### 4.4 XGBoost Classifier

**O que faz**: Treina um modelo de gradient boosting otimizado.

**Definição**: XGBoost é uma implementação eficiente de gradient boosting que geralmente oferece o melhor desempenho em dados tabulares.

**Vantagens**:
- Estado da arte para dados tabulares
- Regularização built-in
- Rápido e eficiente

**Caso de uso**: Modelo de alta performance para competições e produção.

In [ ]:
print("="*80)
print("🚀 XGBOOST CLASSIFIER")
print("="*80)

# Calculando scale_pos_weight para desbalanceamento
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Treinando XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    eval_metric='logloss'
)

print("\n🔄 Treinando XGBoost...")
xgb_model.fit(X_train, y_train, verbose=False)
print("✅ Treinamento concluído!")

# Predições
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Métricas
print("\n📊 MÉTRICAS DE PERFORMANCE:")
print(f"  • Accuracy:  {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"  • Precision: {precision_score(y_test, y_pred_xgb):.4f}")
print(f"  • Recall:    {recall_score(y_test, y_pred_xgb):.4f}")
print(f"  • F1-Score:  {f1_score(y_test, y_pred_xgb):.4f}")
print(f"  • ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Baixa Severidade', 'Alta Severidade']))

### 4.5 Comparação de Modelos

**O que faz**: Compara a performance dos três modelos para selecionar o melhor.

**Caso de uso**: Decisão informada sobre qual modelo levar para produção.

In [ ]:
# Criando DataFrame de comparação
models_comparison = pd.DataFrame({
    'Modelo': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_pred_proba_lr),
        roc_auc_score(y_test, y_pred_proba_rf),
        roc_auc_score(y_test, y_pred_proba_xgb)
    ]
})

print("="*80)
print("📊 COMPARAÇÃO DE MODELOS - Classificação de Severidade")
print("="*80)
print("\n" + models_comparison.to_string(index=False))

# Identificando melhor modelo
best_model_idx = models_comparison['F1-Score'].idxmax()
best_model_name = models_comparison.loc[best_model_idx, 'Modelo']
best_f1 = models_comparison.loc[best_model_idx, 'F1-Score']

print(f"\n🏆 MELHOR MODELO: {best_model_name}")
print(f"   F1-Score: {best_f1:.4f}")

### 4.6 Matriz de Confusão e Curva ROC

**O que faz**: Visualiza a performance do melhor modelo através de matriz de confusão e curva ROC.

**Definição**:
- Matriz de Confusão: Mostra verdadeiros positivos, falsos positivos, verdadeiros negativos e falsos negativos
- Curva ROC: Mostra o trade-off entre taxa de verdadeiros positivos e falsos positivos

**Caso de uso**: Comunicar performance para stakeholders técnicos e não-técnicos.

In [ ]:
# Usando o melhor modelo (assumindo XGBoost)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Matriz de Confusão
cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Baixa Severidade', 'Alta Severidade'],
            yticklabels=['Baixa Severidade', 'Alta Severidade'])
axes[0].set_title('Matriz de Confusão - XGBoost', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Valor Real', fontsize=12)
axes[0].set_xlabel('Valor Predito', fontsize=12)

# 2. Curva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_proba_xgb)

axes[1].plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={roc_auc_score(y_test, y_pred_proba_lr):.3f})', linewidth=2)
axes[1].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={roc_auc_score(y_test, y_pred_proba_rf):.3f})', linewidth=2)
axes[1].plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC={roc_auc_score(y_test, y_pred_proba_xgb):.3f})', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)

axes[1].set_xlabel('Taxa de Falsos Positivos', fontsize=12)
axes[1].set_ylabel('Taxa de Verdadeiros Positivos', fontsize=12)
axes[1].set_title('Curvas ROC - Comparação de Modelos', fontsize=14, fontweight='bold', pad=15)
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Visualizações de avaliação geradas!")

### 4.7 Feature Importance (Interpretabilidade)

**O que faz**: Identifica quais features são mais importantes para as predições do modelo.

**Definição**: Feature importance quantifica a contribuição de cada feature para as predições do modelo.

**Caso de uso**: 
- Entender drivers de risco
- Validar se o modelo está aprendendo padrões sensatos
- Simplificar modelos removendo features irrelevantes
- Comunicar insights para gestão de risco

In [ ]:
# Feature Importance do XGBoost
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Top 15 features
top_15_features = feature_importance.head(15)

print("="*80)
print("🔍 FEATURE IMPORTANCE - Top 15 Features Mais Importantes")
print("="*80)
print("\n" + top_15_features.to_string(index=False))

# Visualização
plt.figure(figsize=(12, 8))
plt.barh(top_15_features['Feature'], top_15_features['Importance'], color='steelblue')
plt.xlabel('Importância', fontsize=12, fontweight='bold')
plt.title('Top 15 Features Mais Importantes - XGBoost\nClassificação de Severidade', 
          fontsize=14, fontweight='bold', pad=20)
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Interpretação para Gestão de Risco:")
print("As features mais importantes indicam os principais fatores")
print("que determinam se um incidente será de alta severidade.")
print("\nGestores devem focar monitoramento e controles nestas áreas.")

## 🎯 5. MODELO 2: Predição de Investigação Regulatória

**Objetivo**: Prever se um incidente resultará em investigação regulatória.

**Variável target**: `regulatory_investigation` (0 = não, 1 = sim)

**Caso de uso real**: 
- Antecipar necessidade de preparar documentação
- Alocar recursos jurídicos e de compliance
- Priorizar remediação de incidentes

In [ ]:
# Preparando dados para Modelo 2
# Usando severidade como feature agora
X2 = df_encoded.drop(['severity_level', 'severity_binary', 'severity_multiclass'], axis=1)

# Adicionando severity_binary como feature
X2['high_severity'] = df_encoded['severity_binary']

# Target é regulatory_investigation (já está no dataset)
# Precisamos garantir que não está em X2
if 'regulatory_investigation' in X2.columns:
    y2 = X2['regulatory_investigation']
    X2 = X2.drop('regulatory_investigation', axis=1)
else:
    y2 = df_encoded['regulatory_investigation']  # Se não estiver, buscar do df original

print("="*80)
print("📊 PREPARAÇÃO PARA MODELO 2: Predição de Investigação Regulatória")
print("="*80)
print(f"\n📋 Dimensão de X2 (features): {X2.shape}")
print(f"🎯 Dimensão de y2 (target): {y2.shape}")
print(f"\n📊 Distribuição do target:")
print(y2.value_counts())
print(f"\n⚖️ Proporção:")
print(f"  Sem investigação: {(y2==0).sum()/len(y2)*100:.1f}%")
print(f"  Com investigação: {(y2==1).sum()/len(y2)*100:.1f}%")

In [ ]:
# Split treino/teste
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=y2
)

print("\n✂️ Split realizado:")
print(f"  Treino: {len(X2_train)} | Teste: {len(X2_test)}")

# Treinando XGBoost (melhor modelo do Modelo 1)
scale_pos_weight_2 = (y2_train == 0).sum() / (y2_train == 1).sum()

xgb_model_2 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_2,
    random_state=RANDOM_STATE,
    eval_metric='logloss'
)

print("\n🔄 Treinando modelo de investigação regulatória...")
xgb_model_2.fit(X2_train, y2_train, verbose=False)
print("✅ Treinamento concluído!")

# Predições
y2_pred = xgb_model_2.predict(X2_test)
y2_pred_proba = xgb_model_2.predict_proba(X2_test)[:, 1]

# Métricas
print("\n📊 MÉTRICAS DE PERFORMANCE:")
print(f"  • Accuracy:  {accuracy_score(y2_test, y2_pred):.4f}")
print(f"  • Precision: {precision_score(y2_test, y2_pred):.4f}")
print(f"  • Recall:    {recall_score(y2_test, y2_pred):.4f}")
print(f"  • F1-Score:  {f1_score(y2_test, y2_pred):.4f}")
print(f"  • ROC-AUC:   {roc_auc_score(y2_test, y2_pred_proba):.4f}")

print("\n📋 Classification Report:")
print(classification_report(y2_test, y2_pred, target_names=['Sem Investigação', 'Com Investigação']))

## 💾 6. Salvamento dos Modelos

**O que faz**: Salva os modelos treinados para uso na API RESTful.

**Caso de uso**: Modelos serão carregados pela API para fazer predições em tempo real.

In [ ]:
import os

# Criando diretório para modelos
os.makedirs('models', exist_ok=True)

# Salvando modelos
joblib.dump(xgb_model, 'models/severity_classifier.pkl')
joblib.dump(xgb_model_2, 'models/investigation_classifier.pkl')

# Salvando lista de features (importante para API)
joblib.dump(X.columns.tolist(), 'models/features_severity.pkl')
joblib.dump(X2.columns.tolist(), 'models/features_investigation.pkl')

print("="*80)
print("💾 MODELOS SALVOS COM SUCESSO")
print("="*80)
print("\n📁 Arquivos criados:")
print("  • models/severity_classifier.pkl")
print("  • models/investigation_classifier.pkl")
print("  • models/features_severity.pkl")
print("  • models/features_investigation.pkl")
print("\n✅ Modelos prontos para deploy na API!")

## 📋 7. Resumo Final da Modelagem

**O que foi realizado neste notebook**:

✅ Preparação de features com encoding e normalização  
✅ Modelo 1: Classificação de Severidade  
   - 3 algoritmos testados (Logistic Regression, Random Forest, XGBoost)  
   - XGBoost com melhor performance  
✅ Modelo 2: Predição de Investigação Regulatória  
   - XGBoost otimizado para desbalanceamento  
✅ Avaliação completa com múltiplas métricas  
✅ Feature importance para interpretabilidade  
✅ Salvamento de modelos para produção  

**Próximos passos**:

➡️ **Notebook 4**: Desenvolvimento da API RESTful com Flask/FastAPI

In [ ]:
print("="*80)
print("🎉 NOTEBOOK 3 CONCLUÍDO COM SUCESSO!")
print("="*80)
print("\n📊 Modelos Desenvolvidos:")
print("\n1️⃣ Classificador de Severidade")
print(f"   • Melhor modelo: XGBoost")
print(f"   • F1-Score: {f1_score(y_test, y_pred_xgb):.4f}")
print(f"   • ROC-AUC: {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")
print("\n2️⃣ Preditor de Investigação Regulatória")
print(f"   • Modelo: XGBoost")
print(f"   • F1-Score: {f1_score(y2_test, y2_pred):.4f}")
print(f"   • ROC-AUC: {roc_auc_score(y2_test, y2_pred_proba):.4f}")
print("\n💼 Aplicação Prática:")
print("   • Priorização de resposta a incidentes")
print("   • Estimativa de risco operacional")
print("   • Alocação de recursos de compliance")
print("   • Suporte à tomada de decisão em gestão de risco")
print("\n➡️ Prossiga para o Notebook 4: API RESTful")